# Shadow-Net · SOREL-20M — FASE 0: VALIDACIÓN DE ACCESO (solo validación, sin entrenamiento)

Master — este notebook **no entrena nada**. Su único objetivo es demostrar de forma reproducible que:

1. El bucket público SOREL-20M es accesible desde Kaggle (HTTPS + S3 unsigned, región `us-west-2`).
2. El servidor acepta `Range Requests` (`206 Partial Content`).
3. `train-features.npz` es un ZIP real inspeccionable por cola (EOCD), sin descargarlo.
4. Se puede extraer **un vector EMBER real de 2381 dims** por streaming, sin materializar 7M×2381.
5. `len == 2381`, sin NaN/Inf, y con etiqueta `is_malware` trazable a `meta.db`.
6. **No se asume** que el orden de `meta.db` coincida con el de `train-features.npz` (se verifica).
7. Cada overlay-feature candidata tiene definición determinista o se descarta (N sin fijar).

Contrato: `X = [EMBER_2381 | OVERLAY_N]` — las 2381 originales nunca se modifican, reordenan ni sustituyen.
Orden canónico local (`extractors/extractor.py:BLOCK_RANGES`):
ByteHistogram 0:256 · ByteEntropy 256:512 · Strings 512:616 · General 616:626 · Header 626:688 · Section 688:943 · Imports 943:2223 · Exports 2223:2351 (total 2381).

Si cualquier celda FAIL → abortar. No pasar a selección 7M, scaler ni FFNN.

In [ ]:
# CELDA 0 — Detección de entorno (sin asumir P100 / 12h / cuota). Solo observa y decide.
import os, shutil, subprocess, sys

def sh(cmd):
    try: return subprocess.check_output(cmd, shell=True, text=True, timeout=15).strip()
    except Exception as e: return f"NA ({e})"

print("python:", sys.version.split()[0])
print("nvidia-smi:", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>&1 | head -5"))
print("RAM:", sh("free -g | head -3"))
print("df /kaggle/working:", sh("df -h /kaggle/working 2>&1 | tail -2"))
print("df /tmp:", sh("df -h /tmp 2>&1 | tail -2"))
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), end="")
    if torch.cuda.is_available():
        print(f" | {torch.cuda.get_device_name(0)} | VRAM={torch.cuda.get_device_properties(0).total_memory/2**30:.1f}GB")
    else: print(" | solo CPU — el pipeline debe seguir siendo válido (más lento)")
except ImportError:
    print("torch: no instalado — instalar en fase de entrenamiento, no aquí")

# Puerta de espacio: meta.db pesa ~3.79GB. Solo avisa, no descarga todavía.
free_b = shutil.disk_usage("/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp").free
print(f"espacio libre destino: {free_b/2**30:.1f} GB (meta.db necesita ~3.8GB + margen)")
assert free_b > 6*2**30, "FAIL: sin espacio mínimo para meta.db + trabajo. Abortar."
print("PASS celda 0: entorno caracterizado, sin supuestos de GPU/horas/cuota.")

In [ ]:
# CELDA 1 — Acceso HTTPS/S3 al bucket público (solo HEAD, cero descargas pesadas).
# Hipótesis H1: el bucket sorel-20m es público en us-west-2 y responde sin credenciales.
import urllib.request

BASE = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020"
TARGETS = {
    "meta.db": f"{BASE}/processed-data/meta.db",                       # ~3.79GB sqlite
    "ember_data.mdb": f"{BASE}/processed-data/ember_features/data.mdb", # ~72GB LMDB (NO descargar)
    "train.npz": f"{BASE}/lightGBM-features/train-features.npz",        # ~121GB (NO descargar)
    "missing": "https://github.com/sophos/SOREL-20M/raw/master/shas_missing_ember_features.json",
}

def head(url, timeout=30):
    req = urllib.request.Request(url, method="HEAD")
    r = urllib.request.urlopen(req, timeout=timeout)
    return r.status, dict(r.headers)

for name, url in TARGETS.items():
    try:
        st, h = head(url)
        print(f"{name}: HTTP {st} | len={h.get('Content-Length')} | range={h.get('Accept-Ranges')} | etag={h.get('ETag')}")
        assert st == 200 and h.get("Accept-Ranges") == "bytes", f"FAIL H1 en {name}"
    except Exception as e:
        print(f"{name}: FAIL ({e})")
        raise AssertionError(f"H1 no verificada para {name}: {e}")
print("PASS celda 1 (H1): bucket público accesible, Accept-Ranges: bytes en los tres artefactos.")

In [ ]:
# CELDA 2 — Range Requests reales (hipótesis H2: el servidor honra Range con 206).
# Sin 206 no hay streaming posible y el diseño chunked queda invalidado.
import urllib.request

NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"

def get_range(url, a, b, timeout=30):
    req = urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}"})
    r = urllib.request.urlopen(req, timeout=timeout)
    return r.status, r.headers.get("Content-Range"), r.read()

st, cr, magic = get_range(NPZ, 0, 3)
print("head-range:", st, cr, magic)
assert st == 206, "FAIL H2: el servidor no devolvió 206"
assert magic == b"PK\x03\x04", f"FAIL H2: magia ZIP ausente: {magic!r}"
TOTAL = int(cr.split("/")[1])
print(f"tamaño total declarado: {TOTAL/2**30:.1f} GiB")
st2, cr2, tail_probe = get_range(NPZ, TOTAL-64, TOTAL-1)
assert st2 == 206 and len(tail_probe) == 64, "FAIL H2: Range de cola no honrado"
print("PASS celda 2 (H2): Range HEAD y TAIL honrados con 206. Streaming viable.")

In [ ]:
# CELDA 3 — Inspección ZIP/NPZ por cola (hipótesis H3: el .npz es un ZIP cuyo
# Central Directory + EOCD permiten listar entradas con 1-2 Range GET, sin descargar 121GB).
import io, struct, urllib.request, zipfile

NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"

def get_range(url, a, b, timeout=60):
    req = urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}"})
    return urllib.request.urlopen(req, timeout=timeout).read()

TOTAL = 121046992510  # re-verificado en celda 2 vía Content-Range; se re-deriva abajo si cambia
tail = get_range(NPZ, TOTAL-131072, TOTAL-1)  # últimos 128KB: EOCD + Central Directory final
eocd = tail.rfind(b"PK\x05\x06")
assert eocd != -1, "FAIL H3: EOCD no encontrado en los últimos 128KB"
cd_count, cd_size, cd_off = struct.unpack("<HHII", tail[eocd+8:eocd+8+2+2+4+4][:12] if False else struct.pack("<HHII",0,0,0,0)) if False else (None,None,None)
# parseo EOCD estándar (22 bytes mínimo)
(n_disk, n_cd, n_entries, n_total, cd_size, cd_off, _) = struct.unpack("<HHHHIIH", tail[eocd+4:eocd+22])
print(f"entradas CD: {n_total} | cd_size={cd_size} | cd_off={cd_off}")
cd_blob = get_range(NPZ, cd_off, cd_off+cd_size-1)
buf = io.BytesIO(cd_blob)
# lista entradas del Central Directory
entries = []
pos = 0
while pos < len(cd_blob):
    assert cd_blob[pos:pos+4] == b"PK\x01\x02", f"FAIL H3: firma CD corrupta en offset {pos}"
    (comp, _, _, _, _, csize, usize, nlen, elen, clen, _, _, _, lho) = struct.unpack("<HHHHHHIIIHHHHHII", cd_blob[pos+10:pos+46])
    name = cd_blob[pos+46:pos+46+nlen].decode()
    entries.append({"name": name, "compress": comp, "csize": csize, "usize": usize, "lho": lho})
    pos += 46 + nlen + elen + clen
for e in entries: print(e)
arr = [e for e in entries if e["name"].endswith(".npy")]
assert arr, "FAIL H3: el npz no contiene ningún .npy"
STORED = all(e["compress"] == 0 for e in arr)
print(f"compresión: {'STORED (row-Range posible)' if STORED else 'DEFLATED (row-Range IMPOSIBLE: ver fallback)'} ")
print("PASS celda 3 (H3): inventario ZIP obtenido por Range. Entradas:", [e['name'] for e in entries])

In [ ]:
# CELDA 4 — Un vector EMBER real por streaming (hipótesis H4: row-slice por Range es posible
# SOLO si el método es STORED; si es DEFLATED esta celda debe FALLAR explícitamente y activar
# el plan B: shards Parquet pre-materializados fuera de Kaggle. No fingir éxito.
# También valida len==2381 y ausencia de NaN/Inf (H5)."
import io, struct, urllib.request, numpy as np

NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"

def get_range(url, a, b, timeout=60):
    req = urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}"})
    r = urllib.request.urlopen(req, timeout=timeout)
    assert r.status == 206, f"Range no honrado: {r.status}"
    return r.read()

# 4a. Local File Header de la primera entrada .npy (30 bytes + nombre + extra)
lho = arr[0]["lho"]  # reutiliza 'arr' de la celda 3
lh = get_range(NPZ, lho, lho+29)
assert lh[:4] == b"PK\x03\x04"
(comp, _, _, _, _, _, _, nlen, elen) = struct.unpack("<HHHHHIIIHH", lh[8:30])
data_off = lho + 30 + nlen + elen
print(f"método={comp} (0=STORED,8=DEFLATED) | data_off={data_off} | csize={arr[0]['csize']}")
if comp != 0:
    raise AssertionError("H4 NO verificada: .npy con DEFLATE — row-Range imposible. Activar plan B (shards Parquet). ABORTAR aquí.")

# 4b. Cabecera .npy (magia 10B + header_len) para obtener shape/dtype/fortran.
npy_head = get_range(NPZ, data_off, data_off+127)
assert npy_head[:6] == b"\x93NUMPY"
hlen = struct.unpack("<H", npy_head[8:10])[0]
hdr = get_range(NPZ, data_off, data_off+10+hlen-1)[10:].decode("latin1")
print("npy header:", hdr.strip())
d = eval(hdr)  # formato controlado por numpy; validado arriba por magia
shape, dtype, order = tuple(d["shape"]), np.dtype(d["descr"]), d["fortran_order"]
assert order is False, "FAIL H4: array Fortran — recalcular strides"
n_rows, n_cols = shape
print(f"shape={shape} dtype={dtype}")
assert n_cols == 2381, f"FAIL H4: columnas={n_cols}, se exigen 2381 exactas"
row_bytes = n_cols * dtype.itemsize
payload = data_off + 10 + hlen

# 4c. Fila 0 por Range puro (cero descargas masivas: solo row_bytes).
row0 = np.frombuffer(get_range(NPZ, payload, payload+row_bytes-1), dtype=dtype)
assert row0.shape == (2381,), f"FAIL H4: fila={row0.shape}"
assert np.all(np.isfinite(row0)), "FAIL H5: NaN/Inf en vector real"
print(f"PASS celdas 4 (H4+H5): fila0 real len=2381, finita. min={row0.min():.4g} max={row0.max():.4g} mean={row0.mean():.4g}")
print("Bloques canónicos: hist=row0[0:256] ent=row0[256:512] str=row0[512:616] gen=row0[616:626] ...")

In [ ]:
# CELDA 5 — Cruce SHA256 ↔ meta.db y etiqueta is_malware (hipótesis H6/H7).
# H6: el orden del .npz NO se asume igual al de meta.db (se verifica contra código oficial).
# H7: cada fila es trazable a (sha256, is_malware) vía meta.db + shas_missing_ember_features.json.
# Requiere meta.db COMPLETO local (sqlite no admite Range): descarga única ~3.8GB con reintentos.
import json, os, sqlite3, urllib.request

WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
META = os.path.join(WORK, "meta.db")
MISSING_URL = "https://github.com/sophos/SOREL-20M/raw/master/shas_missing_ember_features.json"

print("Referencia oficial: sophos/SOREL-20M dataset.py Dataset(mode='train') filtra")
print("  WHERE rl_fs_t <= train_validation_split y key = sha256 ascii → LMDB/NPZ. El .npz se")
print("  generó con build_numpy_arrays_for_lightgbm.py iterando ese Dataset; el ORDEN del .npz")
print("  es el orden de iteración del generador, NO garantizado igual a 'SELECT ... FROM meta'.")
print("→ Por eso esta celda EXIGE el fichero de correspondencia o aborta.")

if not os.path.exists(META):
    print(f"descargando meta.db → {META} (una vez, ~3.8GB)...")
    req = urllib.request.Request("http://sorel-20m.s3.amazonaws.com/09-DEC-2020/processed-data/meta.db")
    with urllib.request.urlopen(req, timeout=60) as r, open(META, "wb") as f:
        while True:
            b = r.read(8*2**20)
            if not b: break
            f.write(b)
print("meta.db bytes:", os.path.getsize(META))
con = sqlite3.connect(META)
cols = [r[1] for r in con.execute("PRAGMA table_info(meta)").fetchall()]
print("columnas meta:", cols)
assert {"sha256", "is_malware", "rl_fs_t"} <= set(cols), "FAIL H7: esquema meta.db inesperado"
n_train = con.execute("SELECT COUNT(*) FROM meta WHERE rl_fs_t <= 1543449600").fetchone()[0]
print(f"filas split train oficial (rl_fs_t<=1543449600): {n_train}")
row = con.execute("SELECT sha256,is_malware,rl_fs_t FROM meta WHERE rl_fs_t <= 1543449600 LIMIT 1").fetchone()
print("ejemplo train:", row[0][:16], "...", "is_malware=", row[1])
con.close()
print("NOTA H6: el .npz NO incluye shas. Sin mapa sha↔índice, la fila0 de la celda 4 NO es")
print("atribuible a un sha concreto. La validación completa exige generar ese mapa fuera de")
print("Kaggle (una vez, con LMDB local) o usar shards Parquet con columna sha256. Si no existe")
print("el mapa, esta fase queda en FAIL controlado — que es el resultado correcto de validar.")

In [ ]:
# CELDA 6 — Candidatas overlay: validación matemática ANTES de fijar N (hipótesis H8).
# Regla: solo entra en OVERLAY_N lo derivable deterministamente del vector EMBER-2381
# (sin pe_metadata, sin binario). Cada candidata declara: fórmula, rango, degeneración.
import numpy as np

# row0 viene de la celda 4 (vector real). Si se corre aislado, usar stub dokumentado:
v = globals().get("row0", None)
if v is None:
    print("AVISO: sin row0 real (celda 4 no ejecutada aquí); se muestran definiciones sin valores.")
    v = np.zeros(2381, dtype=np.float32)

sec, gen = v[688:943], v[616:626]  # SectionInfo 255-D, General 10-D (orden canónico)
cands = {
    "section_count_proxy": ("nº secciones con size>0 (SecInfo incluye sizes). Rango [0,15]. DETERMINISTA.", True),
    "high_entropy_sec": ("nº secciones entropía>7.2 (SecInfo incluye entropía/sección). Rango [0,15]. DETERMINISTA.", True),
    "virt_raw_growth": ("ratio virtual/raw agregado (SecInfo incluye vsize+rawsize). Rango [0,∞), clip 999. DETERMINISTA.", True),
    "cert_table_present": ("DataDirectory CERTIFICATE size>0 — EMBER-v2 SOREL incluye datadirectories en Header/General. A CONFIRMAR contra features.py v2.", None),
    "debug_dir_present": ("DataDirectory DEBUG size>0. Igual condición que cert. A CONFIRMAR.", None),
    "overlay_ratio_proxy": ("NO DERIVABLE del vector 2381: requiere tamaño fichero + fin última sección (pe_metadata/binario). RECHAZADA como proxy puro.", False),
    "overlay_entropy": ("NO DERIVABLE: requiere bytes del overlay. RECHAZADA sin binario.", False),
}
for k, (desc, ok) in cands.items():
    print(f"[{'OK' if ok else ('?' if ok is None else 'NO')}] {k}: {desc}")
print("\nConclusión H8: N NO se fija hoy. Pasan el filtro solo las 3 primeras (puras de SecInfo).")
print("cert/debug quedan PENDIENTES de confirmar el layout EMBER-v2 de SOREL.")
print("Nombres/rangos publicados no son prueba: hay que mapear indices exactos.")


In [ ]:
# CELDA 7 — VEREDICTO (puerta dura). Todo PASS → luz verde a fase 7M. Un FAIL → abortar.
checks = {
    "H1 bucket público + Accept-Ranges": True,   # celda 1
    "H2 Range 206 + magia PK": True,             # celda 2
    "H3 inventario ZIP por cola": True,          # celda 3
    "H4 row-slice STORED 2381": None,            # celda 4: True si STORED, False si DEFLATED
    "H5 finitud + 2381 exactas": None,           # celda 4
    "H6 orden npz!=meta verificado": False,      # celda 5: sin mapa sha-indice NO atribuible -> FAIL controlado
    "H7 etiqueta trazable": False,               # celda 5: exige mapa o shards con sha
    "H8 overlay determinista (N sin fijar)": True, # celda 6
}
# Trasladar resultados reales: si la celda 4 paso, marcar H4/H5 True antes de evaluar.
try:
    assert globals().get("row0", None) is not None and len(row0) == 2381
    checks["H4 row-slice STORED 2381"] = True
    checks["H5 finitud + 2381 exactas"] = True
except Exception:
    pass
for k, val in checks.items():
    s = "PASS" if val is True else ("PEND" if val is None else "FAIL")
    print("[" + s + "] " + k)
fails = [k for k, val in checks.items() if val is not True]
if fails:
    raise SystemExit("VEREDICTO: ABORTAR - pendiente/falla: " + str(fails) + ". No pasar a 7M/scaler/FFNN.")
print("VEREDICTO: LUZ VERDE a seleccion 7M estratificada (seed fija, split train oficial, sin leakage).")
